# Plan-Execute + Map-Reduce — обзор рынка

Комбинация двух паттернов: **Plan-Execute** создаёт стратегический план из нескольких шагов, а когда один из шагов требует массовой обработки — включается **Map-Reduce**. Планировщик определяет ключевых игроков рынка, затем `Send()` запускает параллельный fan-out: каждая компания анализируется отдельным аналитиком. После агрегации результатов управление возвращается планировщику для финального шага — написания сравнительного отчёта.

```mermaid
graph LR
    START((START)) --> identify{{"🎯 Identify<br/><i>планирует задачи</i>"}}
    identify -->|"Send() fan-out"| analyst1(["🔹 Analyst 1<br/><i>анализирует часть</i>"])
    identify -->|"Send() fan-out"| analyst2(["🔹 Analyst 2<br/><i>анализирует часть</i>"])
    identify -->|"Send() fan-out"| analystN(["🔹 Analyst N<br/><i>анализирует часть</i>"])
    analyst1 --> aggregate(["📊 Aggregate<br/><i>объединяет</i>"])
    analyst2 --> aggregate
    analystN --> aggregate
    aggregate --> report(["🔹 Report<br/><i>пишет отчёт</i>"])
    report --> END((END))

    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef aggregator fill:#F59E0B,stroke:#D97706,color:#fff,stroke-width:2px
    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff

    class START,END terminal
    class identify coordinator
    class analyst1,analyst2,analystN,report worker
    class aggregate aggregator
```

In [1]:
import sys
sys.path.insert(0, "..")

import json
import operator
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Два класса состояний

`CompanyState` — вход для каждого параллельного аналитика: имя компании и контекст задачи. `State` — основное состояние графа. Ключевое поле — `analyses` с редьюсером `operator.add`: каждый аналитик возвращает список из одного элемента, и LangGraph автоматически конкатенирует их в общий список, а не перезаписывает. Поле `phase` управляет маршрутизацией между фазами плана.

In [3]:
llm = get_llm()

# --- Состояние для параллельного анализа (вход Map) ---
class CompanyState(TypedDict):
    company: str
    context: str

# --- Основное состояние графа ---
class State(TypedDict):
    goal: str
    companies: list[str]
    analyses: Annotated[list[str], operator.add]
    report: str
    phase: str

## Фаза 1: определить компании (Plan)

Первый шаг плана — планировщик просит LLM назвать ключевых игроков в указанной области. Ответ ожидается в формате JSON-списка. Если LLM вернёт невалидный JSON, срабатывает fallback с тремя дефолтными компаниями — типичная защита от нестабильности формата ответов.

In [4]:
def identify_companies(state: State) -> dict:
    response = llm.invoke([
        SystemMessage(content="Назови 3-4 ключевые компании/продукта в указанной области. Верни JSON-список: [\"name1\", \"name2\", ...]"),
        HumanMessage(content=state["goal"])
    ])
    try:
        companies = json.loads(response.content.strip())
    except json.JSONDecodeError:
        companies = ["LangGraph", "CrewAI", "AutoGen"]
    print(f"  [Фаза 1] Компании: {companies}")
    return {"companies": companies, "phase": "analyze"}

## Фаза 2: Map-Reduce — параллельный анализ компаний

Здесь включается Map-Reduce. Функция `spawn_analysts` возвращает список объектов `Send()` — по одному на каждую компанию. LangGraph запускает узел `analyze_company` параллельно для каждого элемента. Каждый аналитик получает на вход `CompanyState` (не основной `State`) и возвращает результат через редьюсер `analyses`.

После завершения всех параллельных аналитиков управление переходит в узел `aggregate` — это фаза Reduce, где результаты собираются воедино.

In [5]:
def spawn_analysts(state: State) -> list[Send]:
    """Fan-out: запуск параллельного анализа каждой компании через Send()."""
    print(f"  [Фаза 2] Map: запускаю {len(state['companies'])} аналитиков")
    return [
        Send("analyze_company", {"company": c, "context": state["goal"]})
        for c in state["companies"]
    ]

def analyze_company(state: CompanyState) -> dict:
    """Анализ одной компании — вызывается параллельно для каждого Send()."""
    response = llm.invoke([
        SystemMessage(content="Дай краткий анализ компании/продукта: сильные стороны, слабые, перспективы. 3-4 предложения."),
        HumanMessage(content=f"Контекст: {state['context']}\n\nКомпания: {state['company']}")
    ])
    print(f"  [Аналитик] {state['company']} — готово")
    return {"analyses": [f"**{state['company']}**: {response.content}"]}

## Reduce: агрегация результатов

Узел `aggregate` — точка синхронизации. LangGraph вызывает его один раз после того, как все параллельные аналитики завершат работу. К этому моменту все результаты уже собраны в `state["analyses"]` благодаря редьюсеру `operator.add`. Узел просто переключает фазу на `report`.

In [6]:
def aggregate(state: State) -> dict:
    """Reduce: агрегация результатов всех параллельных аналитиков."""
    print(f"  [Фаза 2] Reduce: агрегация {len(state['analyses'])} анализов")
    return {"phase": "report"}

## Фаза 3: финальный отчёт

Последний шаг плана — синтез. LLM получает все собранные анализы и пишет сравнительный обзор: лидеры, тренды, рекомендации. Результат записывается в `state["report"]`.

In [7]:
def write_report(state: State) -> dict:
    """Финальный шаг: сравнительный обзор на основе всех анализов."""
    all_analyses = "\n\n".join(state["analyses"])
    response = llm.invoke([
        SystemMessage(content="Напиши сравнительный обзор (5-6 предложений). Выдели лидеров, тренды, рекомендации."),
        HumanMessage(content=f"Цель: {state['goal']}\n\nАнализы:\n{all_analyses}")
    ])
    print(f"  [Фаза 3] Отчёт готов")
    return {"report": response.content, "phase": "done"}

## Сборка графа

Граф линеен на верхнем уровне (identify -> aggregate -> report -> END), но между identify и aggregate находится параллельный слой: условное ребро из `identify` вызывает `spawn_analysts`, которая возвращает список `Send()`. Каждый `Send()` создаёт отдельный экземпляр узла `analyze_company`. Все экземпляры сходятся в `aggregate` через безусловное ребро.

In [8]:
graph = StateGraph(State)
graph.add_node("identify", identify_companies)
graph.add_node("analyze_company", analyze_company)
graph.add_node("aggregate", aggregate)
graph.add_node("report", write_report)

graph.add_edge(START, "identify")
graph.add_conditional_edges("identify", spawn_analysts)
graph.add_edge("analyze_company", "aggregate")
graph.add_edge("aggregate", "report")
graph.add_edge("report", END)

app = graph.compile()

## Запуск

Передаём цель — сравнительный обзор фреймворков для мультиагентных систем. Граф пройдёт три фазы: определит компании, параллельно проанализирует каждую, напишет итоговый отчёт.

In [9]:
print("=" * 60)
print("ПРИМЕР: Plan-Execute + Map-Reduce — обзор рынка")
print("=" * 60)

result = app.invoke({
    "goal": "Сравнительный обзор фреймворков для мультиагентных систем",
    "companies": [], "analyses": [], "report": "", "phase": "identify",
})

print(f"\n  Проанализировано: {len(result['companies'])} компаний")
print(f"  Отчёт: {result['report'][:300]}...")

ПРИМЕР: Plan-Execute + Map-Reduce — обзор рынка


  [Фаза 1] Компании: ['LangGraph', 'AutoGen', 'CrewAI', 'Semantic Kernel']
  [Фаза 2] Map: запускаю 4 аналитиков


  [Аналитик] LangGraph — готово
  [Аналитик] CrewAI — готово
  [Аналитик] AutoGen — готово


  [Аналитик] Semantic Kernel — готово
  [Фаза 2] Reduce: агрегация 4 анализов


  [Фаза 3] Отчёт готов

  Проанализировано: 4 компаний
  Отчёт: Лидерами рынка выглядят **LangGraph** и **AutoGen**, при этом LangGraph сильнее в production-grade оркестрации, а AutoGen — в гибкости и быстром прототипировании сложных сценариев. **CrewAI** выделяется самым низким порогом входа и хорошо подходит для прикладных мультиагентных workflows, но пока уст...


## Итоги

Комбинация Plan-Execute + Map-Reduce решает задачу в три фазы:

1. **Plan** — планировщик определяет набор объектов для анализа (компании, продукты, источники)
2. **Map** — `Send()` запускает параллельный fan-out, каждый аналитик работает независимо; **Reduce** — агрегация через редьюсер `operator.add`
3. **Execute** — финальный шаг плана синтезирует собранные анализы в итоговый отчёт

Ключевые приёмы:
- **Два класса состояний**: `State` (основной граф) и `CompanyState` (вход для параллельных узлов) — `Send()` принимает произвольный словарь, не обязательно совпадающий с основным состоянием
- **Условное ребро + `Send()`**: `add_conditional_edges` вызывает функцию, которая возвращает список `Send()`, а не строку с именем узла
- **Автоматическая синхронизация**: все параллельные ветки сходятся в `aggregate` без явного барьера — LangGraph ждёт завершения всех `Send()` перед переходом к следующему узлу